# Environnement

In [1]:
from t_dev_810.config import Config, PROCESS_TYPE, MODEL_TYPE, DatasetPaths
from PIL import Image
from t_dev_810.utils.utils import flatten_images
from t_dev_810.utils.versionning import extract_dataset_process


config = Config(PROCESS=[],PROCESSED=extract_dataset_process(), MESSAGE="code refacto", IMG_SIZE=(64, 64), MODEL=MODEL_TYPE.logistic_regression)
dataset_paths = DatasetPaths.load_from_dataset()

X_train = [Image.open(path).convert("L") for path in dataset_paths.train_normal_paths + dataset_paths.train_pneumonia_paths]
y_train = [0] * len(dataset_paths.train_normal_paths) + [1] * len(dataset_paths.train_pneumonia_paths)
X_val = [Image.open(path).convert("L") for path in dataset_paths.val_normal_paths + dataset_paths.val_pneumonia_paths]
y_val = [0] * len(dataset_paths.val_normal_paths) + [1] * len(dataset_paths.val_pneumonia_paths)
X_test = [Image.open(path).convert("L") for path in dataset_paths.test_normal_paths + dataset_paths.test_pneumonia_paths]
y_test = [0] * len(dataset_paths.test_normal_paths) + [1] * len(dataset_paths.test_pneumonia_paths)

print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_val)}")
print(f"Number of test samples: {len(X_test)}")


Number of training samples: 4185
Number of validation samples: 1047
Number of test samples: 234


## Flatten Image

In [2]:

X_train = flatten_images(X_train)
X_val = flatten_images(X_val)
X_test = flatten_images(X_test)

# Compute 

## Logistic Regression

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV


param_grid = [
    {
        "solver": ["lbfgs"],
        "max_iter": [2000],
        "l1_ratio": [0], 
        "C": [0.01, 0.1, 1, 10],
        "class_weight": [None, "balanced"]
    },
    {
        "solver": ["liblinear"],
        "max_iter": [2000],
        "l1_ratio": [0, 1],  
        "C": [0.01, 0.1, 1, 10],
        "class_weight": [None, "balanced"]
    },
    {
        "solver": ["saga"],
        "max_iter": [6000],
        "l1_ratio": [0, 0.5, 1],  
        "C": [0.01, 0.1, 1],
        "class_weight": [None, "balanced"]
    }
]


if PROCESS_TYPE.grid_search in config.PROCESS:
    if config.MODEL == MODEL_TYPE.logistic_regression:
        print("Performing grid search...")
        grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=3, n_jobs=-1, verbose=3)
        grid_search.fit(X_train + X_val, y_train+y_val)
        print("Best parameters: ", grid_search.best_params_)
        model = grid_search.best_estimator_
    else : 
        raise ValueError("Grid search is only implemented for logistic regression")
    
else:
    model = LogisticRegression(max_iter=config.MAX_ITER) if config.MODEL == MODEL_TYPE.logistic_regression else None
    if model is None:
        raise ValueError("Model not implemented")

    model.fit(X_train, y_train)

y_pred = model.predict(X_test)


# Result Visualization

# Register

In [4]:
import pandas as pd
from t_dev_810.utils.versionning import evaluate_model

evaluate_model(config, model, X_train, y_train, X_test, y_test, y_pred)

/Users/nicolasdambreville/Dev/Epitech/T-DEV-810/src/t_dev_810/utils/versionning.py:59: RuntimeWarning: invalid value encountered in scalar divide
  recall = tp / (fn + tp)
/Users/nicolasdambreville/Dev/Epitech/T-DEV-810/.venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## Compare

### Details :

- **accuracy**: proportion of correct predictions made by the model.

- **cv_auc**: AUC computed using 5-fold cross-validation on the training set.  
  It helps estimate how well the model separates the two classes.  
  The standard deviation indicates how stable the model is across different splits of the data.

- **recall**: measures the ability of the model to correctly detect pneumonia cases.  
  In this context, recall is important because false negatives (predicting normal when the patient has pneumonia) are dangerous.  
  Therefore, it is better to have more false positives than false negatives.

- **precision**: measures how many predicted pneumonia cases are actually correct.  
  It is useful to compare with recall because a model with very high recall may produce many false positives.

- **f1_score**: harmonic mean of precision and recall.  
  It balances the two metrics and penalizes models that perform poorly on one of them.

- **test_auc**: measures the ability of the model to distinguish between classes independently of the classification threshold. if cv_aux is highter and test_auc is lower, we have a case of overfitting. 

- **processed**: list all process used on the dataset

In [5]:
from typing import Any, Dict
import json
from datetime import datetime

results: Dict[str, Any]  = {}

with open("version.json", "r") as f:
    version_data = json.load(f)["versions"]
    for date, data in version_data.items():
        key_date = datetime.fromisoformat(date)
        results[key_date] = {
            "accuracy": f"{data['results']['accuracy']:.3f}",
            "cv_auc": f"{data['results']['cv_auc'][0]:.3f} ± {data['results']['cv_auc'][1]:.3f}",
            "recall": f"{data['results']['recall']:.3f}",
            "precision": f"{data.get('results', {}).get('precision', 0):.3f}",
            "f1_score": f"{data.get('results', {}).get('f1_score', 0):.3f}",
            "test_auc": f"{data['results']['test_auc']:.3f}",
            "processed": f"‘{', '.join(data['envs']['PROCESS'])}’",
        }

df = pd.DataFrame.from_dict(results, orient="index")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
print(df)

                           accuracy         cv_auc recall precision f1_score  \
2026-03-03 14:28:07.336452    0.915  0.974 ± 0.001  0.945     0.000    0.000   
2026-03-03 14:42:06.082988    0.742  0.987 ± 0.003  0.982     0.000    0.000   
2026-03-03 14:46:41.773931    0.747  0.989 ± 0.002  0.987     0.000    0.000   
2026-03-03 14:53:41.147665    0.747  0.989 ± 0.002  0.987     0.000    0.000   
2026-03-03 15:58:38.571501    0.924  0.978 ± 0.002  0.957     0.000    0.000   
2026-03-03 16:05:43.158186    0.745  0.989 ± 0.002  0.987     0.714    0.829   
2026-03-03 17:13:22.787274    0.737  0.988 ± 0.001  0.985     0.708    0.824   
2026-03-04 15:40:51.474641    0.795  0.985 ± 0.002  0.979     0.761    0.857   
2026-03-04 16:46:32.898948    0.795  0.985 ± 0.002  0.979     0.761    0.857   
2026-03-04 16:58:30.586683    0.795  0.985 ± 0.002  0.979     0.761    0.857   
2026-03-04 17:56:49.185695    0.750  0.986 ± 0.002  0.987     0.718    0.832   
2026-03-09 13:29:55.348274    0.750  0.9

=> PCA pour réduire la dimension car over fitting meme après grid search 
=> commentaire entre chaque expérience sur résultat hypothèse pour résoudre le prob
=> grid search => il faut y ajouter le test et la val car c'est lui qui gère direct la séparation val et train 